# Chemical Representation II - Graph representation
In this lab, we learn how to represent a molecule with graphs.
:
The package we use is PyTorch Geometric (PyG), a subpackage developed on top of PyTorch.
Google colab has torch installed. In other platforms, you will need to install PyTorch first `pip install torch`.

In [20]:
# install required packages
!pip install -q torch-geometric
!pip install -q rdkit

In [21]:
import torch
from torch_geometric.data import Data

## CO2 example with basic graph encoding
We first only include the atomic numbers and edge index.

In [22]:
# 1. define nodes with atomic numbers
x = torch.tensor([[8], [6], [8]], dtype=torch.float) # note each node is a list of one number here.

# 2. Edge index: [Start list, End list]
# We have 2 bonds between (0,1) and (1,2), but we include both directions for an undirected graph:
# 0-1 (O=C), 1-0 (C=O), 1-2 (C=O), 2-1 (O=C)
edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.int64) # indices are integers

data = Data(x=x, edge_index=edge_index) # initiate a Data object

print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges // 2}") # divided by 2 due to repeated edge index
print("Feature matrix: \n", data.x)
print("Edge index: \n", data.edge_index)

Number of nodes: 3
Number of edges: 2
Feature matrix: 
 tensor([[8.],
        [6.],
        [8.]])
Edge index: 
 tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])


### Adding Chemical Features
Let's first add node features -> feature matrix



In [23]:
x = torch.tensor([
    [8, 2, 0],  # Oxygen 0: [O, sp2, neutral]
    [6, 1, 0],  # Carbon 1: [C, sp1, neutral]
    [8, 2, 0]   # Oxygen 2: [O, sp2, neutral]
], dtype=torch.float)

# Edge index remains the same as connectivity doesn't change
edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.int64)

data = Data(x=x, edge_index=edge_index)

print("Feature matrix: \n", data.x)
print("Edge index: \n", data.edge_index)

Feature matrix: 
 tensor([[8., 2., 0.],
        [6., 1., 0.],
        [8., 2., 0.]])
Edge index: 
 tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])


Next we add more features to the edges (adding bond type and conjugation).

In [24]:
# 1. Node features: [Atomic Number, Hybridization, Charge]
x = torch.tensor([
    [8, 2, 0],  # Oxygen 0
    [6, 1, 0],  # Carbon 1
    [8, 2, 0]   # Oxygen 2
], dtype=torch.float)

# 2. Edge index: [Start list, End list]

edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.long)

# 3. Edge attributes: [Bond Type, Is Conjugated]
# Bond Type: 2.0 for double bond
# Is Conjugated: 1.0 for yes, 0.0 for no (in CO2, the pi system is conjugated)
edge_features = [2.0, 1.0]

edge_attr = torch.tensor([
    edge_features, # Features for edge 0 -> 1
    edge_features, # Features for edge 1 -> 0
    edge_features, # Features for edge 1 -> 2
    edge_features  # Features for edge 2 -> 1
], dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# Check the shapes
print(f"Node feature shape: {data.x.shape}")         # [3, 3]
print(f"Edge index shape:   {data.edge_index.shape}") # [2, 4]
print(f"Edge feature shape: {data.edge_attr.shape}")  # [4, 2]


print("Feature matrix: \n", data.x)
print("Edge index: \n", data.edge_index)
print("Edge attributes: \n", data.edge_attr)

Node feature shape: torch.Size([3, 3])
Edge index shape:   torch.Size([2, 4])
Edge feature shape: torch.Size([4, 2])
Feature matrix: 
 tensor([[8., 2., 0.],
        [6., 1., 0.],
        [8., 2., 0.]])
Edge index: 
 tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
Edge attributes: 
 tensor([[2., 1.],
        [2., 1.],
        [2., 1.],
        [2., 1.]])


## Making it automated
 Define a function to generate the graph representation of an arbitrary molecule

In [25]:
from rdkit import Chem

def molecule_to_graph(smiles):
    # 1. Parse SMILES to RDKit molecule object
    # Chem.AddHs(mol) can be used if you want to include Hydrogens as nodes
    mol = Chem.MolFromSmiles(smiles)

    if mol is None: # bad molecule
      print("Error: Invalid SMILES string")
      return None

    # 2. Extract Node Features (X)
    # Features: [Atomic Number, Hybridization (int), Formal Charge]
    node_features = []
    for atom in mol.GetAtoms():
        atom_features = [
            atom.GetAtomicNum(),
            int(atom.GetHybridization()),
            atom.GetFormalCharge()
        ]
        node_features.append(atom_features) # append is a used to add elements to a list: list_name.append(new_member)

    x = torch.tensor(node_features, dtype=torch.float)

    # 3. Extract Edge Index and Edge Features
    edge_indices_start = []
    edge_indices_end = []
    edge_attrs = []

    for bond in mol.GetBonds():
        # Get indices of the two atoms connected by the bond
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        # Bond features: [Bond Type (float), Is Conjugated (0 or 1)]
        bond_features = [
            bond.GetBondTypeAsDouble(), # Double here means a float number
            1.0 if bond.GetIsConjugated() else 0.0
        ]

        # Add the above to the lists
        # Edge i -> j
        edge_indices_start.append(i)
        edge_indices_end.append(j)
        edge_attrs.append(bond_features)

        # Edge j -> i
        edge_indices_start.append(j)
        edge_indices_end.append(i)
        edge_attrs.append(bond_features)

    # Convert lists to tensors
    # edge_index must be shape [2, E]
    edge_index = torch.tensor([edge_indices_start, edge_indices_end], dtype=torch.int64)
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    # 4. Create PyG Data Object
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# Testing with CO2
smiles_co2 = "O=C=O"
data = molecule_to_graph(smiles_co2)

print("CO2 Graph Representation:")
print(f"Node Features (AtomicNum, Hybrid, Charge):\n{data.x}")
print(f"\nEdge Index (Source -> Target):\n{data.edge_index}")
print(f"\nEdge Attributes (BondType, Conjugated):\n{data.edge_attr}")

CO2 Graph Representation:
Node Features (AtomicNum, Hybrid, Charge):
tensor([[8., 3., 0.],
        [6., 2., 0.],
        [8., 3., 0.]])

Edge Index (Source -> Target):
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])

Edge Attributes (BondType, Conjugated):
tensor([[2., 1.],
        [2., 1.],
        [2., 1.],
        [2., 1.]])


Uh oh, RDKit gave wrong numbers for CO2!

This is just a different convention with RDKit, `GetHybridization()` is topology-based, it determines hybridization roughly as:

| # total valence | hybridization |
| --------------- | ------------- |
| 2               | sp            |
| 3               | sp²           |
| 4               | sp³           |


You need to be careful when using RDKit in the future!

However, as long as you have a consistent feature matrix, you should be fine.

Your turn! Try out some more molecules!